# Panel Regression — Economic Impact Dashboard

Ridge regression and panel data models to quantify environment-to-economy
linkages across regions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

## Modeling Steps

1. Load panel feature matrix
2. Define target variable (GDP growth or crop yield)
3. Fit Ridge regression with cross-validation (TimeSeriesSplit)
4. Estimate region fixed effects
5. Evaluate model performance (R2, MAE)

In [ ]:
# Load panel data
panel = pd.read_parquet('../data/processed/panel_features.parquet')

# Define features and target
target = 'gdp_index'
feature_cols = [c for c in panel.columns if c not in ['date', 'region', target]]
X = panel[feature_cols].values
y = panel[target].values

print(f'Features: {len(feature_cols)}')
print(f'Samples: {len(y)}')

In [ ]:
# Ridge regression with time-series CV
tscv = TimeSeriesSplit(n_splits=5)
alphas = np.logspace(-2, 4, 20)

ridge_cv = RidgeCV(alphas=alphas, cv=tscv)
ridge_cv.fit(X, y)
print(f'Best alpha: {ridge_cv.alpha_:.4f}')
print(f'R2 score: {ridge_cv.score(X, y):.4f}')

# Feature importance
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': ridge_cv.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print('\nTop 10 features by |coefficient|:')
print(coef_df.head(10).to_string(index=False))